In [0]:
def write_user_post_interaction_table(spark, run_date, db, table_names): 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{user_post_interaction} PARTITION (partition_date = '{run_date}')
              with user_set as (
                select distinct customer_id as user_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer
                where coalesce(to_date(from_unixtime(update_at / 1000)),to_date(from_unixtime(create_at / 1000))) <= '{run_date}'
                and delete_type = 0
                and `identity` != 2
                and agree_community_time is not null
              ),
              show_posts as (
                select distinct user_id, content_id as post_id, 'impression' as interaction
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_card_show_i_d
                where page_name in ('community_recommend','community_keyword', ' community_search_moment', 'community_moment', 'community_group_detail_moment')
                and element = 'content_card'
                and user_id in (select user_id from user_set)
                and dt = date_sub('{run_date}',1)
              ),
              view_posts as (
                select distinct customer_id as user_id, post_id, 'view' as interaction
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_browse
                where to_date(from_unixtime(browse_time/1000)) = '{run_date}'
                and customer_id in (select user_id from user_set)
              ),
              like_posts as (
                select distinct customer_id as user_id, post_id, 'like' as interaction
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_like
                where to_date(from_unixtime(like_time/1000)) = '{run_date}'
                and delete_at is null
                and customer_id in (select user_id from user_set)
              ),
              share_posts as (
                select distinct user_id, content_id as post_id, 'share' as interaction
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_click_main_i_d
                where page_name in ('community_content_share', 'community_recommend_share','commnuity_moment_share', 'commnunity_group_moment_share')
                and element = 'share'
                and user_id in (select user_id from user_set)
                and dt = date_sub('{run_date}',1)
              ),
              comment_posts as (
                select distinct customer_id as user_id, post_id, 'comment' as interaction
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_comment
                where to_date(from_unixtime(content_time/1000)) = '{run_date}'
                and delete_at is null
                and customer_id in (select user_id from user_set)
              ),
              publish_posts as (
                select distinct customer_id as user_id, `id` as post_id, 'publish' as interaction
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post
                where to_date(from_unixtime(post_time/1000)) = '{run_date}'  
                and audit_status != 2 --审核状态：0-审核中，1-已通过，2-未过审，3-待人工
                and is_hide = 0  
                and publish_approval_status not in (0, 4) --发布批准状态：0-待提交，1-待审核，2-已通过，3-已发布，4-已驳回
                and delete_at is null
                and customer_id in (select user_id from user_set)
              )
              select *
              from show_posts
              union all
              select *
              from view_posts
              union all
              select *
              from like_posts
              union all
              select *
              from share_posts
              union all
              select *
              from comment_posts
              union all
              select *
              from publish_posts
              """.format(db = db,
                         user_post_interaction = table_names["user_post_interaction"],
                         run_date = run_date))

In [0]:
import json

# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
write_user_post_interaction_table(spark, run_date, db, table_names)